In [6]:
import pandas as pd
df = pd.read_csv("data/sample_transactions.csv")

df["date"] = pd.to_datetime(df["date"])
df["date"].dtype

df["date"].dt.month

pd.to_datetime(df["date"], errors="coerce")

test = pd.Series(["2026-01-01", "not a date"])
pd.to_datetime(test, errors="coerce")

test = pd.DataFrame({
    "date": ["2026-01-01", "not a date", "2026-01-03"],
    "amount": [100, 200, "abc"]
})

test["date"] = pd.to_datetime(test["date"], errors="coerce")
test["amount"] = pd.to_numeric(test["amount"], errors="coerce")
test

test["date"].isna()

bad = test["date"].isna() | test["amount"].isna()
bad

test[bad]     # the rejected rows
test[~bad]    # the good rows

test2 = pd.DataFrame({
    "description": ["Swiggy", "  ", None, "Netflix"],
    "amount": [-400, -100, -50, -649]
})

test2["description"] = test2["description"].str.strip()
test2

missing = test2["description"].isna() | (test2["description"] == "")
missing

test2.loc[missing, "description"] = "(no description)"
test2

test3 = pd.DataFrame({
    "date": ["2026-09-12", "2026-09-12", "2026-09-13"],
    "description": ["Swiggy Order", "Swiggy Order", "Swiggy Order"],
    "amount": [-450, -450, -450]
})

test3.duplicated(subset=["date", "description", "amount"])

is_dup = test3.duplicated(subset=["date", "description", "amount"])
test3[~is_dup]

is_dup.sum()

np.int64(1)

In [1]:
import os
import pandas as pd
os.chdir(r"C:\Users\Hari Prasath\moneymap")

from src.data_loader import load_transactions
from src.data_cleaning import clean_transactions
from src.categorization import add_categories
from src.analytics import calculate_summary

raw = load_transactions("data/sample_transactions.csv")
clean, rejected, summary = clean_transactions(raw)
clean = add_categories(clean)
start = pd.Timestamp("2026-08-01")
end = pd.Timestamp("2026-08-31")

filtered = clean[(clean["date"] >= start) & (clean["date"] <= end)]

print(len(filtered))
calculate_summary(filtered)

from src.analytics import filter_transactions

food = filter_transactions(clean, "2026-07-01", "2026-09-30", ["Food Delivery"])
print(len(food))
food["amount"].sum()

overrides = {"Udemy Course": "Entertainment"}

test_clean = clean.copy()
test_clean["category"] = test_clean["description"].map(overrides).fillna(test_clean["category"])

test_clean.loc[test_clean["description"] == "Udemy Course", ["description", "category"]]

from src.categorization import apply_overrides

fixed = apply_overrides(clean, {"Udemy Course": "Entertainment", "Mystery": "Shopping"})
fixed["category"].value_counts()["Entertainment"]

from src.database import connect, save_override, load_overrides
from src.categorization import apply_overrides

db = connect(":memory:")
save_override(db, "Udemy Course", "Entertainment")
save_override(db, "Udemy Course", "Shopping")     # changes our mind
print(load_overrides(db))

fixed = apply_overrides(clean, load_overrides(db))
fixed.loc[fixed["description"] == "Udemy Course", "category"].iloc[0]

17
9
{'Udemy Course': 'Shopping'}


'Shopping'